# CS 281 Final Experiments — Is Chain-of-Thought Just Rationalization?

End-to-end pipeline for the final report. Runs on Colab/A100 (Llama-3-8B fp16).
Scale knobs live in the Config cell.

## 1. Setup

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q',
                    'transformer-lens>=2.0', 'datasets>=2.18',
                    'huggingface-hub>=0.22', 'accelerate>=0.28', 'seaborn'], check=True)

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv()
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN not set. Local: .env. Colab: Secret named HF_TOKEN.'
print('HF_TOKEN loaded:', os.environ['HF_TOKEN'][:6] + '...')

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.isdir('cot-faithfulness-audit'):
        subprocess.run(['git', 'clone', 'https://github.com/bballhaus/cot-faithfulness-audit.git'], check=True)
    os.chdir('/content/cot-faithfulness-audit')
sys.path.insert(0, os.path.abspath('.'))
print('cwd:', os.getcwd(), '| package present:', os.path.isdir('cot_faithfulness'))

In [ ]:
import json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from cot_faithfulness import (data, model as model_mod, prompts, generation,
                              logit_lens, tuned_lens, patching, analysis, experiments)

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
torch.set_grad_enabled(False)
sns.set_context('notebook'); sns.set_style('whitegrid')

## 2. Config

In [ ]:
CFG = dict(
    model_name = model_mod.DEFAULT_MODEL,
    N_BOOLQ    = 1500,
    N_MNLI     = 1000,
    TAU        = 0.8,
    TAUS       = (0.5, 0.7, 0.8, 0.9),
    MAX_LEN    = 1024,
    COT_TOKENS = 180,
    TUNED_FIT_N = 96,
    TUNED_EVAL_N = 250,
    QUAL_K     = 25,
    SEED       = 42,
)
print(json.dumps(CFG, indent=2, default=str))

## 3. Load model

In [ ]:
model = model_mod.load_model(name=CFG['model_name'])
device = next(model.parameters()).device
N_LAYERS = model.cfg.n_layers
print(f'Loaded {model.cfg.model_name} on {device} | n_layers={N_LAYERS} | d_vocab={model.cfg.d_vocab}')

## 4. Load data

In [ ]:
boolq = data.load_boolq(n=CFG['N_BOOLQ'], seed=CFG['SEED'])
mnli  = data.load_mnli(n=CFG['N_MNLI'], seed=CFG['SEED'])
print('BoolQ:', len(boolq), '| MNLI:', len(mnli))

## 5. Commitment: no-CoT baseline vs with-CoT (BoolQ)

A rationalization account predicts the with-CoT commitment distribution shifts
earlier (shallower) than the no-CoT baseline.

In [ ]:
recs_nocot, probs_nocot = experiments.run_commitment(
    model, boolq, 'boolq', tau=CFG['TAU'], with_cot=False, max_len=CFG['MAX_LEN'])
pd.DataFrame(recs_nocot).to_csv(RESULTS / 'boolq_commitment_nocot.csv', index=False)
np.save(RESULTS / 'boolq_probs_nocot.npy', probs_nocot)
len(recs_nocot)

In [ ]:
recs_cot, probs_cot = experiments.run_commitment(
    model, boolq, 'boolq', tau=CFG['TAU'], with_cot=True,
    max_len=CFG['MAX_LEN'], max_new_tokens=CFG['COT_TOKENS'])
pd.DataFrame(recs_cot).to_csv(RESULTS / 'boolq_commitment_cot.csv', index=False)
np.save(RESULTS / 'boolq_probs_cot.npy', probs_cot)
len(recs_cot)

In [ ]:
summ_nocot = analysis.commitment_summary_ci(recs_nocot)
summ_cot   = analysis.commitment_summary_ci(recs_cot)
print('NO-COT  :', json.dumps(summ_nocot, indent=2, default=float))
print('WITH-COT:', json.dumps(summ_cot, indent=2, default=float))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
analysis.plot_commitment_compare(recs_nocot, recs_cot, N_LAYERS, ax=ax)
plt.tight_layout(); plt.savefig(RESULTS / 'commitment_compare.png', dpi=150); plt.show()

In [ ]:
def mean_pcorrect(probs, recs):
    labels = np.array([r['label'] for r in recs])
    return probs[np.arange(len(recs)), :, labels].mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mean_pcorrect(probs_nocot, recs_nocot), label='no-CoT')
ax.plot(mean_pcorrect(probs_cot, recs_cot), label='with-CoT')
ax.axhline(CFG['TAU'], color='red', ls='--', alpha=0.5, label=f"tau={CFG['TAU']}")
ax.set_xlabel('layer'); ax.set_ylabel('mean P(correct class)')
ax.set_title('Logit-lens confidence by depth: no-CoT vs with-CoT (BoolQ)')
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS / 'mean_curve_compare.png', dpi=150); plt.show()

## 6. Threshold sensitivity (tau sweep)

In [ ]:
labels_nocot = np.array([r['label'] for r in recs_nocot])
labels_cot   = np.array([r['label'] for r in recs_cot])
sweep_nocot = analysis.threshold_sweep(probs_nocot, labels_nocot, taus=CFG['TAUS'])
sweep_cot   = analysis.threshold_sweep(probs_cot,   labels_cot,   taus=CFG['TAUS'])
sweep_nocot['prompt'] = 'no-CoT'; sweep_cot['prompt'] = 'with-CoT'
sweep = pd.concat([sweep_nocot, sweep_cot], ignore_index=True)
sweep.to_csv(RESULTS / 'threshold_sweep.csv', index=False)
sweep

## 7. Qualitative extremes (with-CoT commitment)

In [ ]:
qual = analysis.qualitative_extremes(recs_cot, k=CFG['QUAL_K'], text_key='cot_text')
qual.to_csv(RESULTS / 'qualitative_extremes.csv', index=False)
print('early-commit example:\n', qual[qual.group=='early'].iloc[0]['cot_text'][:400])
print('\nlate-commit example:\n', qual[qual.group=='late'].iloc[0]['cot_text'][:400])
qual[['idx','commitment_layer','correct','group']]

## 8. Tuned-lens robustness comparison

Train per-layer affine translators on a calibration set, then recompute
commitment with both lenses on held-out examples.

In [ ]:
rng = random.Random(CFG['SEED'])
fit_idx = rng.sample(range(len(boolq)), min(CFG['TUNED_FIT_N'], len(boolq)))
fit_seqs = []
for i in fit_idx:
    ex = data.boolq_example(boolq[i])
    toks = model.to_tokens(prompts.format_boolq_direct(ex['passage'], ex['question']))
    if toks.shape[1] <= CFG['MAX_LEN']:
        fit_seqs.append(toks)
lens = tuned_lens.fit_tuned_lens(model, fit_seqs, rank=256, steps=250, lr=1e-3)
print('fitted tuned lens on', len(fit_seqs), 'calibration prompts')

In [ ]:
eval_idx = [i for i in range(len(boolq)) if i not in set(fit_idx)][:CFG['TUNED_EVAL_N']]
bt = prompts.boolq_target_tokens(model); tids = [bt['No'], bt['Yes']]
ll_layers, tl_layers = [], []
for i in tqdm(eval_idx, desc='tuned vs logit'):
    ex = data.boolq_example(boolq[i])
    toks = model.to_tokens(prompts.format_boolq_direct(ex['passage'], ex['question']))
    if toks.shape[1] > CFG['MAX_LEN']:
        continue
    pos = [toks.shape[1] - 1]
    p_ll = logit_lens.logit_lens(model, toks, tids, positions=pos)[:, 0, :]
    p_tl = tuned_lens.tuned_lens_probs(model, lens, toks, tids, positions=pos)[:, 0, :]
    ll_layers.append(logit_lens.commitment_layer(p_ll, ex['label'], CFG['TAU']))
    tl_layers.append(logit_lens.commitment_layer(p_tl, ex['label'], CFG['TAU']))
lens_cmp = {
    'logit_lens_mean_commit': analysis.bootstrap_ci([x for x in ll_layers if x >= 0]),
    'tuned_lens_mean_commit': analysis.bootstrap_ci([x for x in tl_layers if x >= 0]),
    'logit_frac_committed': float(np.mean(np.array(ll_layers) >= 0)),
    'tuned_frac_committed': float(np.mean(np.array(tl_layers) >= 0)),
}
print(json.dumps(lens_cmp, indent=2, default=float))
json.dump(lens_cmp, open(RESULTS / 'tuned_vs_logit.json', 'w'), indent=2, default=float)

## 9. CoT corruption + pre-CoT patching (MNLI)

Random-token corruption is the baseline; semantic inversion is the primary
causal probe. The CoT span excludes the answer line and scoring is at the
post-CoT / pre-answer position.

In [ ]:
mnli_random = experiments.run_corruption(
    model, mnli, 'mnli', strategy='random', do_patch=True,
    max_len=CFG['MAX_LEN'], max_new_tokens=160)
pd.DataFrame(mnli_random).to_csv(RESULTS / 'mnli_corruption_random.csv', index=False)
print('random usable =', len(mnli_random))

In [ ]:
mnli_invert = experiments.run_corruption(
    model, mnli, 'mnli', strategy='invert', do_patch=True,
    max_len=CFG['MAX_LEN'], max_new_tokens=160)
pd.DataFrame(mnli_invert).to_csv(RESULTS / 'mnli_corruption_invert.csv', index=False)
print('invert usable =', len(mnli_invert))

In [ ]:
sum_random = analysis.patching_summary(mnli_random, with_ci=True)
sum_invert = analysis.patching_summary(mnli_invert, with_ci=True)
print('RANDOM:', json.dumps(sum_random, indent=2, default=float))
print('INVERT:', json.dumps(sum_invert, indent=2, default=float))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, recs, name in [(axes[0], mnli_random, 'random'), (axes[1], mnli_invert, 'invert')]:
    df = pd.DataFrame(recs)
    sns.histplot(df['logit_diff_drop'], bins=30, ax=ax)
    fr = df['flipped'].mean()
    pr = df['patch_recovers_clean'].mean() if 'patch_recovers_clean' in df else float('nan')
    ax.set_title(f'{name}: flip={fr:.2f}, patch-recovery={pr:.2f}')
    ax.set_xlabel('clean P(ans) - corrupted P(ans)')
plt.tight_layout(); plt.savefig(RESULTS / 'mnli_corruption.png', dpi=150); plt.show()

## 10. Save final summary

In [ ]:
artifact = {
    'config': {k: str(v) for k, v in CFG.items()},
    'model_name': model.cfg.model_name,
    'n_layers': N_LAYERS,
    'boolq_commitment_nocot': summ_nocot,
    'boolq_commitment_cot': summ_cot,
    'threshold_sweep': sweep.to_dict(orient='records'),
    'tuned_vs_logit': lens_cmp,
    'mnli_corruption_random': sum_random,
    'mnli_corruption_invert': sum_invert,
}
json.dump(artifact, open(RESULTS / 'final_summary.json', 'w'), indent=2, default=float)
print('Saved', RESULTS / 'final_summary.json')

## 11. (Optional) Cross-family generalization

Re-run sections 3-5 with a second model to test whether late commitment is
Llama-specific. Alt models use different chat templates, so build prompts via
`prompts.format_chat` instead of the hand-built Llama-3 formatter.